In [ ]:
#Full Code discretly monitored up and out put barrier option with antithetical variance reduction
import time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Math 
import seaborn as sns
import scipy.stats as stats

np.random.seed(42)

#discretly monitored up and out call barrier option
# Model Parameters 
S0 = 100        # Start Price
K = 100         # Strike
B = 150         # Up-and-Out Barrier
T = 1.0         # Duration (In Years)
r = 0.05        # risk-free interst rate
sigma = 0.2     # Volatility
dividend  = 0.03   # Dividend yield (continuous)

# Monte-Carlo-Parameter
n_sim = 100000     # Number of simulations
n_steps = 252       # Number of time discretisations (1 day)
dt = T / n_steps    # Timestep

#GBM in python code with atntithetic variates
Z = np.random.normal(size=(n_sim, n_steps)) #matrix with standard-normally distributed random numbers for use in wiener-process
Z_anti = -Z  # antithetic counterpart

S = np.zeros((n_sim, n_steps)) #structure for all the simulated price history
S_anti = np.zeros((n_sim, n_steps))

S[:, 0] = S0 #every first column of each simulation starts with value S_0
S_anti[:, 0] = S0


for t in range(1, n_steps):
    S[:, t] = S[:, t - 1] * np.exp((r - dividend - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z[:, t]) #GBM equation with dividend 
    S_anti[:, t] = S_anti[:, t - 1] * np.exp((r - dividend - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z_anti[:, t]) #GBM equation with dividend                                                                                                  #Dividdens from Glassman, 2003, P.97

# Barrier check (knockout)
knockout = np.any(S >= B, axis=1)
knockout_anti = np.any(S_anti >= B, axis=1)


#python code for payoff and discounted price
payoff = np.where(knockout, np.maximum(K - S[:, -1], 0.0), 0.0)
payoff_anti = np.where(knockout_anti, np.maximum(K - S_anti[:, -1], 0.0), 0.0)

# Average both payoffs for variance reduction
payoff_avg = 0.5 * (payoff + payoff_anti)

# Discounted Price (Antithetic Estimator)
discounted_price = np.exp(-r * T) * np.mean(payoff_avg)

#stats
all_discounted = np.exp(-r * T) * payoff_avg  #Calculate all discounted individual payoffs
std_dev = np.std(all_discounted, ddof=1)   # standard deviation
standard_error = std_dev / np.sqrt(len(all_discounted))  # Standard error

z = stats.norm.ppf(0.95) #95% confidence interval
ci_low = discounted_price - z * standard_error
ci_high = discounted_price + z * standard_error

#print
# Output
print (f"Discountet Price: {discounted_price:.4f}")
print(f"Standard Deviation of Payoffs: {std_dev:.4f}")
print(f"Standard Error of the Estimator: {standard_error:.6f}")
print(f"95% Confidence Interval: [{ci_low:.4f}, {ci_high:.4f}]")
